In [1]:
import os
import pandas as pd
import numpy as np
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. RUTES I CÀRREGA DE DADES
# ==============================================================================
# Llegeix el teu Excel Master
ruta_dataset = '../data/processed/04_Master_Dataset_Final_Integrat.xlsx'
ruta_logbook = '../data/processed/historial_experiments.csv'

print("📂 Carregant el Master Dataset definitiu...")
df = pd.read_excel(ruta_dataset)

# ==============================================================================
# 2. TAULER DE CONTROL DE VARIABLES (FEATURE SELECTION)
# ==============================================================================
# Aquí pots comentar (posar un #) o descomentar columnes per fer proves
columnes_X = [
    'renda_disponible_Import_Euros', 
    'pad_dom_Llars_1_Avi_Sol', 
    'edificacions_any_Any_Mitja_Ponderat', 
    'heating_thermal_demand_intensity__2025-03-06',
    'torrid_nights__2024-01-01'
]

# La teva variable dependent real
columna_Y = 'CVI__2023-01-01' 

# Neteja de seguretat: eliminem qualsevol fila que tingui un NaN just en aquestes columnes
df_model = df.dropna(subset=columnes_X + [columna_Y]).copy()

X = df_model[columnes_X]
y = df_model[columna_Y]

# --- SANITY CHECK: ANÀLISI DE COLINEALITAT ---
print("\n📊 Calculant correlacions entre les variables d'entrada (X)...")
matriu_corr = X.corr()
# Mirem específicament la relació que et preocupava:
corr_edat_demanda = matriu_corr.loc['edificacions_any_Any_Mitja_Ponderat', 'heating_thermal_demand_intensity__2025-03-06']
print(f"⚠️ Correlació Edat Edifici vs Demanda Tèrmica: {corr_edat_demanda:.2f}")
if abs(corr_edat_demanda) > 0.75:
    print("   -> ALERTA: Estan massa correlacionades. Et recomano treure'n una per a futurs experiments.")

# ==============================================================================
# 3. PREPARACIÓ DELES DADES (TRAIN/TEST & SCALING)
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n📐 Dataset dividit. Train: {X_train.shape[0]} zones | Test: {X_test.shape[0]} zones.")

# ==============================================================================
# 4. FUNCIÓ DEL LOGBOOK (MLOps)
# ==============================================================================
def registrar_experiment(nom_model, hiperparametres, variables_usades, mae, r2):
    ara = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    nou_registre = pd.DataFrame([{
        'Data_Hora': ara,
        'Model': nom_model,
        'Variables_X': str(variables_usades),
        'Hiperparametres': str(hiperparametres),
        'MAE_Test': round(mae, 4),
        'R2_Test': round(r2, 4)
    }])
    
    if not os.path.exists(ruta_logbook):
        nou_registre.to_csv(ruta_logbook, index=False, sep=';')
    else:
        nou_registre.to_csv(ruta_logbook, mode='a', header=False, index=False, sep=';')
    print(f"   [📝 Logbook Actualitzat]")

# ==============================================================================
# 5. ENTRENAMENT 1: RANDOM FOREST
# ==============================================================================
print("\n🌲 Entrenant el Random Forest...")
rf_params = {'n_estimators': 150, 'max_depth': 12, 'random_state': 42}
model_rf = RandomForestRegressor(**rf_params)
model_rf.fit(X_train, y_train) 

preds_rf = model_rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, preds_rf)
r2_rf = r2_score(y_test, preds_rf)

print(f"   -> Resultat RF | MAE: {mae_rf:.2f} | R2: {r2_rf:.3f}")
registrar_experiment("Random Forest", rf_params, columnes_X, mae_rf, r2_rf)

# ==============================================================================
# 6. ENTRENAMENT 2: XARXA NEURONAL (MLP)
# ==============================================================================
print("\n🧠 Entrenant la Xarxa Neuronal (MLP)...")
mlp_params = {
    'hidden_layer_sizes': (128, 64, 32),
    'activation': 'relu',
    'solver': 'adam',
    'max_iter': 800,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'random_state': 42
}

model_mlp = MLPRegressor(**mlp_params)
model_mlp.fit(X_train_scaled, y_train)

preds_mlp = model_mlp.predict(X_test_scaled)
mae_mlp = mean_absolute_error(y_test, preds_mlp)
r2_mlp = r2_score(y_test, preds_mlp)

print(f"   -> Resultat MLP | MAE: {mae_mlp:.2f} | R2: {r2_mlp:.3f}")
registrar_experiment("MLP", mlp_params, columnes_X, mae_mlp, r2_mlp)

print("\n🚀 EXPERIMENT FINALITZAT AMB ÈXIT!")

📂 Carregant el Master Dataset definitiu...

📊 Calculant correlacions entre les variables d'entrada (X)...
⚠️ Correlació Edat Edifici vs Demanda Tèrmica: 0.34

📐 Dataset dividit. Train: 907 zones | Test: 161 zones.

🌲 Entrenant el Random Forest...
   -> Resultat RF | MAE: 3.92 | R2: 0.788
   [📝 Logbook Actualitzat]

🧠 Entrenant la Xarxa Neuronal (MLP)...
   -> Resultat MLP | MAE: 4.20 | R2: 0.753
   [📝 Logbook Actualitzat]

🚀 EXPERIMENT FINALITZAT AMB ÈXIT!
